# Incident risk — high severity incident classifier


## 1. Business Understanding

**PREDICTIVE.** Flag residents at higher **risk of a High-severity** incident (from aggregated history + case features) to prioritize supervision — **not** to blame individuals.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'resident_master.csv')
df['target_high'] = (df['high_severity_count'] > 0).astype(int)
print(df['target_high'].value_counts(normalize=True))


In [ ]:
num = ['days_in_program','incident_frequency','session_count','avg_health_score_trend','counseling_session_count']
cat = ['initial_risk_level','case_category']
for c in ['sub_cat_trafficked','sub_cat_physical_abuse','sub_cat_sexual_abuse']:
    df[c] = df[c].astype(str).str.lower().eq('true').astype(int)
flags = ['sub_cat_trafficked','sub_cat_physical_abuse','sub_cat_sexual_abuse']
m = df[num + cat + flags + ['target_high']].copy()
for c in num:
    m[c] = m[c].fillna(m[c].median())
for c in cat:
    m[c] = m[c].fillna('Unknown').astype(str)


In [ ]:
sns.heatmap(m[num+['target_high']].corr(), annot=True); plt.show()
print(stats.chi2_contingency(pd.crosstab(m['case_category'], m['target_high']))[:2])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
X = m[num + cat + flags]
y = m['target_high']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_f = num + flags
cat_f = cat
prep = ColumnTransformer([
 ('n', Pipeline([('im', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_f),
 ('c', Pipeline([('im', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_f)])


## 3. Modeling & Feature Selection


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
clf = Pipeline([('prep', prep), ('rf', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced'))])
clf.fit(X_train, y_train)
print('AUC', roc_auc_score(y_test, clf.predict_proba(X_test)[:,1]))
fi = clf.named_steps['rf'].feature_importances_
names = clf.named_steps['prep'].get_feature_names_out()
print(pd.Series(fi, index=names).sort_values(ascending=False).head(12))


## 4. Evaluation & Interpretation


In [ ]:
from sklearn.metrics import classification_report, RocCurveDisplay, confusion_matrix
print(classification_report(y_test, clf.predict(X_test), digits=3))
fig, ax = plt.subplots(figsize=(5,4))
RocCurveDisplay.from_predictions(y_test, clf.predict_proba(X_test)[:,1], ax=ax)
plt.show()
print(confusion_matrix(y_test, clf.predict(X_test)))


## 5. Causal / Relationship Analysis

**sklearn LogisticRegression** on dummy-coded train features — **exp(coef)** readout; the RF remains the **operational** model.


In [ ]:
from sklearn.linear_model import LogisticRegression
Xe = pd.get_dummies(X_train, drop_first=True).apply(pd.to_numeric, errors='coerce').fillna(0.0)
lr_ex = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42, solver='lbfgs')
lr_ex.fit(Xe, y_train)
coef = pd.Series(lr_ex.coef_[0], index=Xe.columns)
print(np.exp(coef).sort_values(ascending=False).head(12))


## 6. Deployment Notes

**No .sav (lighter).** Would expose risk badge via API using `joblib` of the `Pipeline` after leadership sign-off; `class_weight='balanced'` mitigates **rare high-severity** prevalence issues but requires human review.
